# 06 - Extra Trees tuning, U.S. training only

Extra Trees is evaluated as an additional U.S. candidate. Its historical combined-data result is not used as a baseline because it came from a different cohort and raw-data protocol. The reviewed environment makes tree searches expensive, so this preliminary run uses five trials and a capped tree range.


In [1]:
from pathlib import Path
import sys, warnings
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'preprocessing.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import randint
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import cross_validate
from utils.us_modeling import (CONSERVATIVE_POLICY, development_cv, load_us_training_data,
    run_randomized_search, scoring, search_result, conservative_pipeline)
SEED = 42
X_train, y_train, project_ids = load_us_training_data(ROOT)
assert len(project_ids) == 26945 and project_ids.is_unique
assert np.isfinite(y_train).all() and set(y_train.unique()) == {0, 1}
assert CONSERVATIVE_POLICY == dict(include_timing=False, include_flags=False, include_reward=False, exclude_renamed=True)
baseline = conservative_pipeline(ExtraTreesClassifier(n_estimators=250, min_samples_leaf=5, n_jobs=1, random_state=SEED), 'target')
base_scores = cross_validate(baseline, X_train, y_train, cv=development_cv(), scoring=scoring(), n_jobs=1, return_train_score=True, error_score='raise')
baseline_row = {m: float(base_scores[f'test_{m}'].mean()) for m in scoring()}
baseline_row['roc_auc_sd'] = float(base_scores['test_roc_auc'].std())
print(f'official U.S. training rows: {len(X_train):,}; success rate: {y_train.mean():.4f}')
print('reference baseline:', {k: round(v, 4) for k, v in baseline_row.items()})


official U.S. training rows: 26,945; success rate: 0.5461
reference baseline: {'roc_auc': 0.7191, 'accuracy': 0.6662, 'f1': 0.7136, 'roc_auc_sd': 0.0053}


In [2]:
model = ExtraTreesClassifier(n_jobs=1, random_state=SEED)
space = {
    'model__n_estimators': randint(100, 301),
    'model__max_depth': [None, 10, 16, 24],
    'model__min_samples_leaf': randint(2, 21),
    'model__min_samples_split': randint(2, 12),
    'model__max_features': ['sqrt', 0.5, 1.0],
    'model__class_weight': [None, 'balanced'],
}
search, trials = run_randomized_search(X_train, y_train, model, 'target', space, n_iter=5)
result = search_result(search, trials, 'Extra Trees')
result['baseline_roc_auc'] = baseline_row['roc_auc']
display(pd.DataFrame([result]).drop(columns=['best_params']).round(4))
print('best parameters:', result['best_params'])
display(trials.head(10).drop(columns=['params']).round(4))


,model,cohort,feature_policy,cv,search_iterations,estimate,cv_roc_auc,cv_roc_auc_sd,train_roc_auc,overfit_gap,accuracy,f1,fit_seconds,elapsed_seconds,baseline_roc_auc
0,Extra Trees,official U.S. training rows,"conservative: no timing, flags, rewards, or re...",5-fold StratifiedKFold shuffle=True random_sta...,5,ordinary tuning CV; optimistic selection estimate,0.7205,0.0061,0.7579,0.0374,0.6685,0.7155,1.8775,74.7021,0.7191


best parameters: {'class_weight': None, 'max_depth': 16, 'max_features': 0.5, 'min_samples_leaf': 20, 'min_samples_split': 8, 'n_estimators': 174}


,trial_rank,cv_roc_auc,cv_roc_auc_sd,train_roc_auc,cv_accuracy,cv_f1,mean_fit_seconds,overfit_gap,elapsed_seconds
0,1,0.7205,0.0061,0.7579,0.6685,0.7155,1.8775,0.0374,74.7021
1,2,0.7203,0.0060,0.7474,0.6683,0.7175,2.0273,0.0271,74.7021
2,3,0.7199,0.0058,0.7987,0.6660,0.7140,2.3794,0.0788,74.7021
3,4,0.7187,0.0057,0.7402,0.6655,0.7159,1.5905,0.0215,74.7021
4,5,0.7187,0.0056,0.7409,0.6636,0.6909,0.8585,0.0222,74.7021


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(trials['params'].map(lambda p: p.get('model__n_estimators')), trials['cv_roc_auc'], c=trials['overfit_gap'], cmap='viridis')
axes[0].set_xlabel('Number of trees'); axes[0].set_ylabel('Validation ROC-AUC'); axes[0].set_title('Extra Trees size and validation score')
axes[1].scatter(trials['mean_fit_seconds'], trials['cv_roc_auc'], c=trials['overfit_gap'], cmap='viridis')
axes[1].set_xlabel('Mean fit seconds'); axes[1].set_ylabel('Validation ROC-AUC'); axes[1].set_title('Fit cost and train-validation gap')
plt.tight_layout()
print(f"reference AUC {baseline_row['roc_auc']:.4f}; tuned AUC {result['cv_roc_auc']:.4f}; difference {result['cv_roc_auc'] - baseline_row['roc_auc']:+.4f}")
print(f"best-trial train-validation gap: {result['overfit_gap']:.4f}")


reference AUC 0.7191; tuned AUC 0.7205; difference +0.0014
best-trial train-validation gap: 0.0374


Extra Trees is retained as a comparable U.S. candidate only if its training-CV score and computational cost justify it. The result is not a final holdout evaluation.
